# What the antenna buys: 21-cm retention under an identical filter

The paper trail for `beam_comparison.pdf`. One sidereal day of the *same* sky
behind the *same* horizon at the *same* site, simulated three times with three
different beams, each pushed through its own eigenmode filter and compared
against the same 21-cm ensemble.

**The question.** `signal_loss.pdf` says the beam-weighted foregrounds are
spectrally low-dimensional and a signal survives filtering that subspace out.
It says nothing about *how much of that is the antenna*, and the manuscript
claims (section~\ref{subsec:eig_antenna}) that the bowtie was designed to
minimise exactly this overlap. That claim has never been measured. This
measures it.

**The three beams.**

| beam | what it is |
|---|---|
| Isotropic | uniform response, still behind the real horizon. The chromaticity-free reference: whatever structure survives here is the *sky's*, not the antenna's. |
| EIGSEP bowtie | the antenna the paper is about. |
| Vivaldi feed | the HERA Phase II feed used in isolation without its dish -- the antenna the October 2024 suspension actually flew. |

**Scope.** Zenith pointing, nominal position, nominal horizon, GSM16, no noise,
no receiver systematics. The only thing that changes between panels is the
beam. This is not a claim about the Vivaldi as an antenna: it is a HERA design
being used well outside the configuration it was optimised for, and EIGSEP
still uses it for the ground antennas, whose job is not a low-chromaticity
global-signal measurement.

**Inputs** (gitignored -- see `../README.md` to regenerate):

| file | from |
|---|---|
| `../output/beam_sims.npz` | `run_beam_sims.py` |
| `../../models_21cm/output/zeus21_models.npz` | `models_21cm/generate.py` |
| `foreground_svd.npz` (paper repo) | the nominal zenith waterfall, as a cross-check |

**Run order: this notebook runs LAST.** It asserts its bowtie panel against
`foreground_svd.npz` and `paper.N_ANCHOR`, and section 6 loads the
substitution values `signal_loss.ipynb` publishes so that the whole draft text
can be written in manuscript order from one template. Run
`horizon_shift.ipynb`, then `signal_loss.ipynb`, then this.

In [1]:
import inspect
import json
import re
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import nbformat as nbf
from matplotlib.lines import Line2D

HERE = Path.cwd()                              # horizon_position/notebooks
sys.path.insert(0, str(HERE.parent))           # paper.py
import paper
sys.path.insert(0, str(paper.ROOT / "models_21cm"))
import selection

PAPER = paper.PAPER
ORDER = paper.BEAM_ORDER                       # isotropic, bowtie, vivaldi
LABELS = paper.BEAM_LABELS
N_SHOW = paper.N_SHOW_BEAM                     # 25: the Vivaldi crosses at ~18
n_modes = np.arange(N_SHOW + 1)

# Identical to signal_loss.ipynb and horizon_shift.ipynb, deliberately: the
# reader has met this ensemble twice already and should recognise the artist.
C_21 = "0.40"

## 1. Inputs

### 1.1 The three waterfalls

`run_beam_sims.py` ran `eigsim.simulate` once per beam at zenith pointing,
nominal position, with the nominal open-sky mask -- the same call
`run_sims.py` makes for its nominal row, with the beam swapped. `t_sys` is
sky + ground pickup + a constant 50 K receiver; `fgnd` is the fractional
ground pickup per channel.

The receiver is subtracted here for the same reason as in `signal_loss.ipynb`:
the eigenbasis is built from the antenna temperature, so a constant it was
never optimised to compress inflates the apparent residual by orders of
magnitude.

In [2]:
d = np.load(paper.BEAM_SIMS_NPZ, allow_pickle=True)
beams = [str(b) for b in d["beams"]]
freqs = d["freqs_mhz"]
t_rcvr = float(d["t_receiver"])
n_time, n_f = d["t_sys"].shape[1], freqs.size
assert set(beams) == set(ORDER), f"beam_sims.npz has {beams}, expected {ORDER}"

t_ant = {b: d["t_sys"][beams.index(b)] - t_rcvr for b in ORDER}
fgnd = {b: d["fgnd"][beams.index(b)] for b in ORDER}

print(f"{n_time} LST samples x {n_f} channels "
      f"({freqs[0]:.0f}-{freqs[-1]:.0f} MHz), lmax = {int(d['beam_lmax'])}")
print(f"sky = {str(d['sky_model'])}, T_ground = {float(d['t_ground']):g} K, "
      f"T_receiver = {t_rcvr:g} K")
print(f"vivaldi source: {str(d['vivaldi_source'])}")
print(f"{str(d['band_limit_note'])}\n")
for b in ORDER:
    print(f"  {LABELS[b]:16s} ground fraction {fgnd[b].mean():.4f}, "
          f"T_ant {t_ant[b].min():7.1f} - {t_ant[b].max():7.1f} K")

1436 LST samples x 201 channels (50-250 MHz), lmax = 128
sky = gsm16, T_ground = 300 K, T_receiver = 50 K
vivaldi source: eigsep_vivaldi.npz


  Isotropic beam   ground fraction 0.6533, T_ant   222.4 -  3258.1 K
  EIGSEP bowtie    ground fraction 0.5540, T_ant   179.4 -  3834.2 K
  Vivaldi feed     ground fraction 0.3048, T_ant   144.1 -  6291.3 K


### 1.2 Sky coupling, and why it is not the story

Each beam sees a different fraction of open sky. The **open-sky fraction**
$\eta = 1 - f_\mathrm{gnd}$ is the beam-weighted share of the sphere that is
not canyon, and it is what an isotropic global signal is multiplied by on its
way into the antenna temperature.

It cuts *against* the conclusion below, which is worth stating plainly: the
Vivaldi is directive enough to put far more of its response on the sky than
the bowtie, so it starts with more signal. The isotropic beam, which has no
directivity at all, does worst of the three -- its $\eta$ is pure geometry.

$\eta$ is also achromatic for the isotropic beam and not for the other two,
which is the cleanest available demonstration that the chromatic swing in
$\eta$ is a property of the *beam* and not of the horizon.

In [3]:
eta = {b: 1.0 - fgnd[b] for b in ORDER}
print(f"{'beam':16s}{'eta mean':>10}{'eta min':>10}{'eta max':>10}{'swing':>9}")
for b in ORDER:
    e = eta[b]
    print(f"{LABELS[b]:16s}{e.mean():>10.3f}{e.min():>10.3f}{e.max():>10.3f}"
          f"{(e.max() - e.min()) / e.mean() * 100:>8.0f}%")

assert np.ptp(eta["isotropic"]) / eta["isotropic"].mean() < 1e-6, (
    "the isotropic beam's open-sky fraction is chromatic, which it cannot be "
    "-- it is pure geometry; the horizon mask or the normalisation changed"
)
print("\nisotropic eta is achromatic, as it must be: the chromatic swing in "
      "the other two is the beam's, not the horizon's")

beam              eta mean   eta min   eta max    swing
Isotropic beam       0.347     0.347     0.347       0%
EIGSEP bowtie        0.446     0.357     0.549      43%
Vivaldi feed         0.695     0.587     0.796      30%

isotropic eta is achromatic, as it must be: the chromatic swing in the other two is the beam's, not the horizon's


### 1.3 The 21-cm ensemble

The same 4096 Zeus21 models and the same posterior reionization cut as the
other two notebooks, asserted against `paper.N_MODELS` so the three figures
cannot cut different ensembles.

**Each beam attenuates the ensemble by its own $\eta$.** Everything here is
uncorrected antenna temperature, so the signal has to be put in that quantity
before it is filtered, and the factor is per-beam. Using one beam's $\eta$ for
all three -- or none at all -- would compare a sky-referred signal against an
antenna-temperature residual.

`T21_sky` is kept un-attenuated for trough depth only, which is a property of
the model rather than of the observation.

In [4]:
m = np.load(paper.MODELS_NPZ, allow_pickle=False)
assert np.array_equal(m["freqs_MHz"], freqs), "21 cm grid != simulation grid"

keep = selection.reionized_across_band(m["xHI"], m["z_xHI"])
assert keep.sum() == paper.N_MODELS, (
    f"{keep.sum()} models survive the cut, expected {paper.N_MODELS} -- "
    "the ensemble moved; update paper.N_MODELS and re-run all three notebooks"
)
T21_sky = m["T21_mK"][keep] * 1e-3              # (n_model, n_f) K, intrinsic
T21 = {b: eta[b] * T21_sky for b in ORDER}      # as each antenna observes it

depth_mK = -T21_sky.min(axis=1) * 1e3
print(f"{keep.sum()} models; intrinsic trough depth median "
      f"{np.median(depth_mK):.0f} mK")
for b in ORDER:
    print(f"  {LABELS[b]:16s} observed median depth "
          f"{np.median(-T21[b].min(axis=1)) * 1e3:5.0f} mK")

1769 models; intrinsic trough depth median 118 mK
  Isotropic beam   observed median depth    41 mK
  EIGSEP bowtie    observed median depth    44 mK
  Vivaldi feed     observed median depth    79 mK


## 2. One eigenbasis per antenna

Each waterfall gets its **own** uncentered SVD, and each ensemble is filtered
in the basis built from the antenna that observed it. That is the only honest
construction -- a beam's foregrounds and the signal it retains are not
separable quantities, and projecting one antenna's models onto another's modes
would compare nothing physical.

It is also why the three panels of the figure cannot share a grey band. Each
band is that antenna's ensemble in that antenna's basis.

In [5]:
def derive(t_ant, eta, t21_sky, n_modes, n_time):
    """Foreground residual and retained-signal percentiles for one antenna.

    Returns ``(fg_resid, t21_pct, t21_resid)`` in K. The ensemble is attenuated
    by this antenna's own open-sky fraction and filtered in its own basis.
    """
    n_f = t_ant.shape[1]
    _, s, Vh = np.linalg.svd(t_ant, full_matrices=False)
    # Parseval: the pooled foreground residual is the singular-value tail.
    tail = np.concatenate([np.cumsum(s[::-1] ** 2)[::-1], [0.0]])
    fg_resid = np.sqrt(tail / (n_time * n_f))[: len(n_modes)]
    coeff = (eta * t21_sky) @ Vh.T
    t21_resid = np.array([np.sqrt(np.sum(coeff[:, N:] ** 2, axis=1) / n_f)
                          for N in n_modes])            # (n_modes, n_model)
    return fg_resid, np.percentile(t21_resid, [5, 50, 95], axis=1), t21_resid


def stays_below(curve, ref, n_modes):
    """Smallest N with curve < ref there and at every larger N on the axis."""
    below = curve < ref
    return next((int(N) for N in n_modes if below[N:].all()), None)

In [6]:
fg_resid, t21_pct, t21_resid, n_anchor = {}, {}, {}, {}
for b in ORDER:
    fg_resid[b], t21_pct[b], t21_resid[b] = derive(
        t_ant[b], eta[b], T21_sky, n_modes, n_time)
    n_anchor[b] = stays_below(fg_resid[b], t21_pct[b][1], n_modes)
    assert n_anchor[b] is not None, (
        f"{b}: the foreground residual never stays below the median retained "
        f"signal within N <= {N_SHOW}; widen paper.N_SHOW_BEAM"
    )

print(f"{'beam':16s}{'N_ANCHOR':>10}{'fg [mK]':>10}{'21cm [mK]':>11}")
for b in ORDER:
    N = n_anchor[b]
    print(f"{LABELS[b]:16s}{N:>10}{fg_resid[b][N] * 1e3:>10.2f}"
          f"{t21_pct[b][1][N] * 1e3:>11.2f}")

beam              N_ANCHOR   fg [mK]  21cm [mK]
Isotropic beam           6      0.14       1.32
EIGSEP bowtie           10      0.62       0.87
Vivaldi feed            18      0.76       0.80


### 2.1 The bowtie panel is Fig. 1

The middle panel is the same simulation `signal_loss.pdf` is built from, so it
has to reproduce it. The raw waterfalls are not bit-identical -- they are
separate `eigsim` runs and differ by ~2 mK on ~200 K -- but that difference
lies in the leading modes, and every quantity the figures quote agrees to
better than a per cent.

If this assert fires, the two figures are showing different bowties.

In [7]:
fg = np.load(paper.FG_NPZ, allow_pickle=True)
assert np.array_equal(fg["freqs_MHz"], freqs), "frequency grid mismatch"
assert np.allclose(fg["fgnd"], fgnd["bowtie"], rtol=1e-6), (
    "beam_sims.npz's bowtie ground fraction differs from foreground_svd.npz"
)
ref_fg, ref_pct, _ = derive(
    fg["t_sys"] - float(fg["t_receiver"]), 1.0 - fg["fgnd"], T21_sky,
    n_modes, fg["t_sys"].shape[0])
for name, got, ref in (("foreground residual", fg_resid["bowtie"], ref_fg),
                       ("median retained", t21_pct["bowtie"][1], ref_pct[1])):
    rel = np.abs(got[:N_SHOW] / ref[:N_SHOW] - 1).max()
    assert rel < 0.01, f"bowtie {name} differs from foreground_svd.npz by {rel:.1%}"
    print(f"bowtie {name:20s} matches foreground_svd.npz to {rel:.2%}")

assert n_anchor["bowtie"] == paper.N_ANCHOR, (
    f"the bowtie panel gives N_ANCHOR = {n_anchor['bowtie']}, but paper.py "
    f"says {paper.N_ANCHOR} -- this figure and Fig. 1 disagree"
)
print(f"\nbowtie N_ANCHOR = {paper.N_ANCHOR}, matching paper.N_ANCHOR")

bowtie foreground residual  matches foreground_svd.npz to 0.00%
bowtie median retained      matches foreground_svd.npz to 0.00%

bowtie N_ANCHOR = 10, matching paper.N_ANCHOR


## 3. What the antenna buys

Two ways of reading it, and they say the same thing.

**Modes.** How deep the filter has to go to reach a given foreground residual.
**Retention at matched suppression.** How much of the 21-cm RMS is left when
each antenna has been filtered to the *same* residual -- the comparison that
conditions correctly. Comparing retention at a fixed *mode count* instead
inverts the answer, because the antenna that has suppressed less naturally
retains more.

In [8]:
def compare(fg_resid, t21_resid, n_modes, order, labels, levels=(10.0, 1.0, 0.1)):
    """Modes to reach a residual level, and the 21 cm retained when it is reached."""
    def modes_to(b, thr):
        return next((int(N) for N in n_modes
                     if (fg_resid[b][N:] * 1e3 < thr).all()), None)

    print(f"{'beam':16s}" + "".join(f"{f'<{t:g} mK':>12}" for t in levels))
    print("modes filtered")
    for b in order:
        row = "".join(f"{modes_to(b, t) or '>' + str(n_modes[-1]):>12}" for t in levels)
        print(f"  {labels[b]:14s}{row}")
    print("21-cm RMS retained there [per cent of unfiltered]")
    for b in order:
        row = ""
        for t in levels:
            N = modes_to(b, t)
            frac = np.median(t21_resid[b][N] / t21_resid[b][0]) * 100 if N else None
            row += f"{f'{frac:.1f}%' if N else '--':>12}"
        print(f"  {labels[b]:14s}{row}")
    return modes_to


modes_to = compare(fg_resid, t21_resid, n_modes, ORDER, LABELS)

ret = {b: np.median(t21_resid[b][modes_to(b, 1.0)] / t21_resid[b][0]) for b in ORDER}
print(f"\nAt a 1 mK foreground residual the bowtie retains "
      f"{ret['bowtie'] / ret['vivaldi']:.1f}x the signal fraction of the Vivaldi, "
      f"and {ret['bowtie'] / ret['isotropic'] * 100:.0f} per cent of what a "
      f"chromaticity-free beam would.")
print(f"It reaches that residual in {modes_to('bowtie', 1.0)} modes against "
      f"{modes_to('vivaldi', 1.0)} for the Vivaldi and "
      f"{modes_to('isotropic', 1.0)} for the isotropic beam.")

assert ret["isotropic"] > ret["bowtie"] > ret["vivaldi"], (
    "the retention ordering changed; the figure's argument is that "
    "chromaticity orders these antennas and the text says so"
)
assert eta["vivaldi"].mean() > eta["bowtie"].mean(), (
    "the Vivaldi no longer couples better to the sky -- the text makes a point "
    "of it losing despite that advantage"
)

beam                  <10 mK       <1 mK     <0.1 mK
modes filtered
  Isotropic beam           4           6           7
  EIGSEP bowtie            8          10          13
  Vivaldi feed            11          18         >25
21-cm RMS retained there [per cent of unfiltered]
  Isotropic beam       33.6%       13.3%       11.6%
  EIGSEP bowtie        16.7%        7.7%        5.3%
  Vivaldi feed          9.9%        3.5%          --

At a 1 mK foreground residual the bowtie retains 2.2x the signal fraction of the Vivaldi, and 58 per cent of what a chromaticity-free beam would.
It reaches that residual in 10 modes against 18 for the Vivaldi and 6 for the isotropic beam.


## 4. The figure

Three panels, one per antenna, each of them exactly the plot the reader has
already met as Fig.~\ref{fig:singular_values}: black foreground residual, grey
5--95 band with a dashed median, log axes, nothing marked at any $N$. The
comparison the reader is asked to make is how far right the black curve
travels before it drops under the band.

Choices worth not re-litigating:

* **Three panels, not three curves on one.** Each antenna has its own
  foreground curve, its own $\eta$ and its own basis, so a single panel would
  put one antenna's foregrounds against another's band. Panels keep every
  quantity beside the antenna it belongs to.
* **A band, not the full ensemble.** Both were rendered. The tails are mild
  (the extremes run a factor of 2--2.5 beyond the band), but 1769 curves have
  no crisp edge, and the edge is what the crossing is read against. The dashed
  median also disappears into the haze.
* **No ratio axis.** A "signal / residual" panel is legible and climbs to
  several hundred, which reads as a sensitivity claim. Same reason there is no
  10 mK line anywhere in this paper.
* **The median is an ensemble statistic, not a model.** It is a different model
  at almost every $N$; a real single model tracks it to within 1.3x over the
  whole axis, so the dashed line is never far from something physical, but the
  caption says "median" and should not say "the signal".

In [9]:
def make_figure(n_modes, fg_resid, t21_pct, order, labels, c_21, path):
    """Three panels of the Fig. 1 axes, one per antenna.

    Shared y so the panels can be read against each other; the label sits
    inside each panel rather than as a title, to keep the row compact enough
    for a two-column `figure*`.
    """
    fig, axs = plt.subplots(1, 3, figsize=(7.0, 2.6), layout="constrained",
                            sharey=True)
    for ax, b in zip(axs, order):
        ax.fill_between(n_modes, t21_pct[b][0], t21_pct[b][2], color=c_21,
                        alpha=0.25, lw=0, zorder=0)
        ax.plot(n_modes, t21_pct[b][1], color=c_21, lw=1.3, ls="--", zorder=1)
        ax.plot(n_modes, fg_resid[b], color="k", lw=1.8, zorder=2)
        ax.set_yscale("log")
        ax.set_xlim(0, n_modes[-1])
        ax.set_ylim(1e-5, 3e3)
        ax.grid(True, which="both", ls=":", lw=0.5, alpha=0.5)
        ax.tick_params(labelsize=7)
        ax.set_xlabel("Foreground modes filtered", fontsize=8)
        ax.text(0.045, 0.94, labels[b], transform=ax.transAxes, va="top",
                fontsize=7.5)
    axs[0].set_ylabel("Residual RMS [K]", fontsize=8)
    # Centre right of the first panel: the white band between the 21 cm
    # ensemble and the foreground curve, which the isotropic residual clears
    # by N ~ 4. Lower right put the box on that residual's descent; upper
    # right collides with the panel label.
    axs[0].legend(
        handles=[
            Line2D([], [], color="k", lw=1.8, label="Beam-weighted foregrounds"),
            Line2D([], [], color=c_21, lw=1.3, ls="--", label="21-cm models"),
        ],
        fontsize=6, loc="center right", framealpha=0.92,
    )
    fig.savefig(path, bbox_inches="tight", dpi=600)
    return fig

In [10]:
fig_beam = make_figure(n_modes, fg_resid, t21_pct, ORDER, LABELS, C_21,
                       PAPER / "beam_comparison.pdf")
print(f"wrote {PAPER / 'beam_comparison.pdf'}")

wrote /home/christian/Documents/research/papers/eigsep_instrument/notebooks/beam_comparison.pdf


## 5. Export to the paper repository

Same convention as the other two: a committed npz (archived to Zenodo) plus a
*standalone* notebook that regenerates the PDF from it and imports nothing
from this repo. The plotting function is lifted out of the running kernel with
`inspect.getsource`, so the archived code is byte-identical to what produced
the figure.

In [11]:
PAPER_MD = r"""
# What the antenna buys: 21-cm retention under an identical filter

One sidereal day of the same GSM16 sky behind the same canyon horizon at the
same site, simulated three times with three different beams. Each antenna's
foregrounds get their own uncentered SVD, and each antenna's 21-cm ensemble is
filtered in that antenna's own basis, so every quantity in a panel belongs to
the antenna named in it.

Both curves in every panel are **antenna temperature**, ground pickup
included. The 21-cm models are multiplied by that antenna's beam-weighted
open-sky fraction before filtering, because an isotropic signal reaches the
antenna temperature attenuated by it. They are not sky-referred amplitudes.

**Why three panels rather than three curves.** Each antenna has a different
foreground residual, a different open-sky fraction and a different eigenbasis.
On one panel the reader would inevitably compare one antenna's foregrounds
against another's ensemble. The grey bands are *not* the same band drawn three
times.

**Why a band rather than every model.** Both were rendered. The ensemble's
extremes run a factor of 2--2.5 beyond the 5--95 band, so little is hidden, but
1769 individual curves have no crisp edge and the crossing -- how far right the
black curve travels before it drops under the grey -- is what the figure is
for. The dashed median is also lost in the haze. The median is an ensemble
statistic and not a single model: it is a different model at almost every $N$,
though a real model tracks it to within 1.3x across the axis.

**The isotropic beam** is the chromaticity-free reference. It still sees the
real horizon, so its open-sky fraction is pure geometry and achromatic; what
little structure survives in its panel is the sky's, not an antenna's. Its
residual leaves the bottom of the frame near $N = 6$, where it reaches machine
precision.

**The Vivaldi feed** is the HERA Phase II design used in isolation without its
dish, which is the antenna EIGSEP's first suspension flew in October 2024. It
couples considerably better to the sky than the bowtie -- it starts with more
signal -- and still retains less of it at matched foreground suppression. This
is not a statement about the feed in the configuration it was designed for.

**Scope.** Zenith pointing, nominal position, GSM16, no noise and no receiver
systematics. Read this as a statement about spectral overlap, not a sensitivity
forecast.

Data: `beam_comparison.npz`. Full derivation from the raw simulation output:
`mock_analysis/horizon_position/notebooks/beam_comparison.ipynb`.
"""

IMPORTS_SRC = """import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D"""

LOAD_SRC = """d = np.load("beam_comparison.npz", allow_pickle=True)
n_modes = d["n_modes"]                 # foreground modes filtered (x-axis)
order = [str(x) for x in d["order"]]   # panel order, left to right
labels = {b: str(l) for b, l in zip(order, d["labels"])}
# Per antenna: foreground residual [K] and the 5/50/95 percentiles of the
# retained 21-cm RMS [K], both already in antenna temperature.
fg_resid = {b: d["fg_resid"][i] for i, b in enumerate(order)}
t21_pct = {b: d["t21_pct"][i] for i, b in enumerate(order)}
C_21 = "0.40"
print(f"{len(order)} antennas, {n_modes[-1]} modes on the axis")"""

CALLS_SRC = """fig = make_figure(n_modes, fg_resid, t21_pct, order, labels, C_21,
                  "beam_comparison.pdf")"""


def export(path, markdown, load_src, funcs, calls):
    """Write a standalone paper-repo notebook: prose, imports, load, code, calls."""
    src = "\n\n\n".join(inspect.getsource(f).rstrip() for f in funcs)
    nb = nbf.v4.new_notebook()
    nb.cells = [
        nbf.v4.new_markdown_cell(markdown.strip()),
        nbf.v4.new_code_cell(IMPORTS_SRC.strip()),
        nbf.v4.new_code_cell(load_src.strip()),
        nbf.v4.new_code_cell(src),
        nbf.v4.new_code_cell(calls.strip()),
    ]
    nbf.write(nb, path)
    print(f"wrote {path}")

In [12]:
np.savez_compressed(
    paper.BEAM_CMP_NPZ,
    n_modes=n_modes,
    order=np.array(ORDER),
    labels=np.array([LABELS[b] for b in ORDER]),
    fg_resid=np.stack([fg_resid[b] for b in ORDER]),
    t21_pct=np.stack([t21_pct[b] for b in ORDER]),
    eta=np.stack([eta[b] for b in ORDER]),
    freqs_MHz=freqs,
    n_anchor=np.array([n_anchor[b] for b in ORDER]),
    n_time=n_time,
    description=(
        "Inputs for the antenna-comparison figure. One sidereal day of the "
        "same GSM16 sky behind the same canyon horizon, simulated once per "
        "beam (order: isotropic / EIGSEP bowtie / Vivaldi feed used without a "
        "dish) at zenith pointing and nominal position. fg_resid (n_beam, "
        "n_modes) [K] is each antenna's foreground residual after filtering "
        "the leading N right singular vectors of ITS OWN uncorrected "
        "antenna-temperature waterfall; t21_pct (n_beam, 3, n_modes) [K] is "
        "the 5/50/95th percentile of the 21 cm ensemble retained under that "
        "same per-antenna filter. Both are antenna temperatures: the models "
        "are multiplied by that antenna's beam-weighted open-sky fraction eta "
        "(n_beam, n_freq) = 1 - fgnd before filtering, so they are not "
        "sky-referred amplitudes. The three bases differ, so the three bands "
        "are not one band repeated. n_anchor is the smallest N past which each "
        "antenna's foreground residual stays below its median retained signal; "
        "the figure marks no dimension. The bowtie column reproduces "
        "foreground_svd.npz, which Fig. 1 is built from. Produced by "
        "mock_analysis/horizon_position/notebooks/beam_comparison.ipynb."
    ),
)
print(f"wrote {paper.BEAM_CMP_NPZ}")

export(
    PAPER / "beam_comparison.ipynb",
    markdown=PAPER_MD,
    load_src=LOAD_SRC,
    funcs=(make_figure,),
    calls=CALLS_SRC,
)

wrote /home/christian/Documents/research/papers/eigsep_instrument/notebooks/beam_comparison.npz
wrote /home/christian/Documents/research/papers/eigsep_instrument/notebooks/beam_comparison.ipynb


## 6. The draft paper text

`paper_text.tex` is a staging file to paste from -- nothing in this repo writes
to the paper's own `.tex` sources. It carries **all five blocks** of the draft,
in manuscript order: the two that come from this notebook and the three that
come from `signal_loss.ipynb`, which publishes its substitution values rather
than writing a file of its own. Every number is substituted from the arrays the
figures are drawn from, so prose and figures cannot drift apart.

One template and one output, because two of each could not put the blocks in
reading order -- blocks 1 and 2 are a single continuous passage split across
the two notebooks -- and because the retired single-panel caption went on
sitting in a file that still looked authoritative.

The template is `../paper_text.tex.in`; its header carries the framing this
text has to preserve, which matters more than the numbers do. In short: the
Vivaldi feed is a HERA design that EIGSEP still uses for the ground antennas
and flew on the first suspension, so the comparison is a documented evolution
and not a rejected candidate, and it is used here outside the configuration it
was optimised for.

In [13]:
LEVEL = 1.0                                    # mK, the suppression compared at


def _sci(x):
    """LaTeX mantissa-exponent form, as signal_loss.ipynb writes DYNRANGE.

    Not f"{log10(x):.0f}" inside $10^{...}$: that rounded 10^5.6 to 10^6 and
    overstated the dynamic range by a factor of 2.5 in the one sentence whose
    job is to quote it.
    """
    e = int(np.floor(np.log10(x)))
    return rf"{x / 10 ** e:.1f}\times10^{{{e}}}"


if not paper.SIGNAL_LOSS_VALS.exists():
    raise SystemExit(
        f"{paper.SIGNAL_LOSS_VALS} not found -- run signal_loss.ipynb first; "
        "blocks 1, 4, 5 and 6 of paper_text.tex.in come from it"
    )
sl_vals = json.loads(str(np.load(paper.SIGNAL_LOSS_VALS)["vals_json"]))

vals = {
    # This figure is Fig. 1 now, so the width the template quotes is its own.
    "FIGW": f"{fig_beam.get_size_inches()[0]:g}",
    "NMODELS": int(paper.N_MODELS),
    "LEVEL": f"{LEVEL:g}",
}
for b, key in zip(ORDER, ("ISO", "BOW", "VIV")):
    N = modes_to(b, LEVEL)
    vals[f"N{key}"] = N
    vals[f"RET{key}"] = f"{np.median(t21_resid[b][N] / t21_resid[b][0]) * 100:.0f}"
    vals[f"NA{key}"] = n_anchor[b]
    vals[f"ETA{key}"] = f"{eta[b].mean():.2f}"
    # Per-beam band extremes. The manuscript quotes eta as a RANGE per antenna,
    # and until 2026-09-02 only the bowtie's came from tokens (ETALO/ETAHI, out
    # of signal_loss.ipynb) while the Vivaldi and isotropic numbers were typed
    # by hand -- the same failure mode that took out the rotation paragraph.
    # ETAISO is quoted as a single value, which is legitimate only because the
    # isotropic eta is achromatic; the assert in section 1.2 is what guarantees
    # that, so do not weaken it.
    vals[f"ETALO{key}"] = f"{eta[b].min():.2f}"
    vals[f"ETAHI{key}"] = f"{eta[b].max():.2f}"
    # Dynamic range at that antenna's own N: how far the filter takes it from
    # the raw waterfall RMS. Block 1 quotes DYNBOW, at NBOW rather than at
    # N_ANCHOR - 1, so the whole passage sits at one mode count and the number
    # answers the "about one part in 10^4" heuristic the manuscript sets up
    # just above it. The numerator is the quantity signal_loss rounds to RMS0.
    vals[f"DYN{key}"] = _sci(fg_resid[b][0] / fg_resid[b][N])
vals["FACTOR"] = f"{ret['bowtie'] / ret['vivaldi']:.1f}"
vals["PCTIDEAL"] = f"{ret['bowtie'] / ret['isotropic'] * 100:.0f}"

# Merge. A key defined by both notebooks must agree, or one of the two is
# quoting a number the other has moved -- except FIGW, which signal_loss no
# longer emits precisely because this figure replaced its own.
clash = {k for k in vals.keys() & sl_vals.keys() if str(sl_vals[k]) != str(vals[k])}
assert not clash, f"signal_loss and beam_comparison disagree on {sorted(clash)}"
merged = {**sl_vals, **vals}

out = (HERE.parent / "paper_text.tex.in").read_text()
for k, v in merged.items():
    out = out.replace(f"@@{k}@@", str(v))
assert "@@" not in out, (
    "unsubstituted token left in paper_text.tex: "
    f"{sorted(set(re.findall(r'@@([A-Z0-9_]+)@@', out)))}"
)
(PAPER / "paper_text.tex").write_text(out)
print(f"wrote {PAPER / 'paper_text.tex'}  "
      f"({len(sl_vals)} values from signal_loss, {len(vals)} from here)\n")
for k in sorted(merged):
    print(f"    {k:10s} {merged[k]}")

wrote /home/christian/Documents/research/papers/eigsep_instrument/notebooks/paper_text.tex  (105 values from signal_loss, 20 from here)

    ABOVE      68
    CLEARUPHI  17
    CLEARUPLO  3
    CLEARUPMID 10
    CLEAR_E    7
    CLEAR_N    4
    CLEAR_U    10
    COMPFG     1.2\times10^{6}
    COMPU      3242
    COSMAX     0.98
    COSMED     0.63
    CVPEN      1.00
    DECFG      11
    DECU       5
    DTU        1.0
    DYNBOW     1.2\times10^{6}
    DYNISO     5.0\times10^{6}
    DYNRANGE   4.0\times10^{5}
    DYNVIV     1.5\times10^{6}
    ETABOW     0.45
    ETAHI      0.55
    ETAISO     0.35
    ETALO      0.36
    ETAMEAN    0.45
    ETAVIV     0.70
    FACTOR     2.2
    FG         0.62
    FGHAND     1.82
    FGM1       1.82
    FIGW       7
    FLOORE1    2.04
    FLOORE10   11
    FLOORN1    1.81
    FLOORN10   2.4
    FLOORU01   1.84
    FLOORU01PCT 1
    FLOORU1    3.52
    FLOORU10   30
    HANDEND    6.2
    HANDFAC    2.5
    HANDPREV   8
    HANDSTAY   12
    KEEPE